In [2]:
from __future__ import annotations

import time
import math
import requests
import numpy as np
import pandas as pd

from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from typing import Dict, Any, List, Optional, Tuple

from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss, accuracy_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression


# -----------------------------
# Config
# -----------------------------
ESPN_BASE = "https://site.web.api.espn.com/apis/site/v2/sports/basketball/nba"

DEFAULT_N_DAYS_HISTORY = 160  # NBA games are frequent; this is plenty
ROLL_WINDOW = 5              # last-5 rolling window
SLEEP_BETWEEN_CALLS = 0.20   # ESPN rate-limit friendliness

# Feature selection (win model)
MIN_NON_NULL_RATE = 0.25
DROP_CONSTANT_FEATURES = True

# Props config
PROP_STATS = {
    "points": "PTS",
    "rebounds": "REB",
    "assists": "AST",
    "threes": "3PM",
    "minutes": "MIN",
}
PROP_TARGETS = ["points", "rebounds", "assists", "threes"]
MIN_PLAYER_GAMES = 8              # minimum history for player to be used
TOP_PLAYERS_PER_TEAM = 8          # show props for top N by recent minutes
PROP_LINE_ROUND_TO = 0.5          # typical prop increments
PROP_STD_FLOOR = 2.0              # avoid degenerate odds if std too low

# -----------------------------
# Odds helpers
# -----------------------------
def to_american_odds(p: float) -> float:
    """Convert win probability p to fair American odds (no vig)."""
    p = float(p)
    p = min(max(p, 1e-6), 1 - 1e-6)
    if p >= 0.5:
        return -100.0 * p / (1.0 - p)
    return 100.0 * (1.0 - p) / p


def safe_float(x: Any) -> float:
    """Convert ESPN-ish values to float when possible; else NaN."""
    if x is None:
        return np.nan
    if isinstance(x, (int, float, np.number)):
        return float(x)
    if isinstance(x, str):
        s = x.strip().replace("%", "")
        if s == "":
            return np.nan
        try:
            return float(s)
        except Exception:
            return np.nan
    if isinstance(x, dict):
        for k in ("value", "displayValue", "stat", "amount"):
            if k in x:
                return safe_float(x[k])
        return np.nan
    return np.nan


def sigmoid(x: float) -> float:
    return 1.0 / (1.0 + math.exp(-x))


def logit(p: float) -> float:
    p = float(p)
    p = min(max(p, 1e-6), 1 - 1e-6)
    return math.log(p / (1.0 - p))


# -----------------------------
# Normal CDF helpers (no SciPy)
# -----------------------------
def norm_cdf(z: float) -> float:
    # standard normal CDF via erf
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))


def prob_over_normal(mean: float, std: float, line: float) -> float:
    std = max(float(std), 1e-6)
    z = (mean - float(line)) / std
    p = norm_cdf(z)
    # P(stat > line) assuming continuous approx
    return min(max(p, 1e-6), 1 - 1e-6)


def round_to_step(x: float, step: float = 0.5) -> float:
    if pd.isna(x):
        return np.nan
    step = float(step)
    return round(float(x) / step) * step


# -----------------------------
# ESPN fetching
# -----------------------------
@dataclass
class ESPNClient:
    sleep: float = SLEEP_BETWEEN_CALLS

    def __post_init__(self):
        self.session = requests.Session()
        self.session.headers.update(
            {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"}
        )

    def get_scoreboard(self, date_yyyymmdd: Optional[str] = None) -> Optional[Dict[str, Any]]:
        url = f"{ESPN_BASE}/scoreboard"
        if date_yyyymmdd:
            url += f"?dates={date_yyyymmdd}"
        try:
            r = self.session.get(url, timeout=20)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"[scoreboard] error for {date_yyyymmdd}: {e}")
            return None

    def get_summary(self, event_id: str) -> Optional[Dict[str, Any]]:
        url = f"{ESPN_BASE}/summary?event={event_id}"
        try:
            r = self.session.get(url, timeout=25)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"[summary] error for event={event_id}: {e}")
            return None

    def get_team_roster(self, team_id: str) -> Optional[Dict[str, Any]]:
        # ESPN public roster endpoint
        url = f"{ESPN_BASE}/teams/{team_id}/roster"
        try:
            r = self.session.get(url, timeout=20)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"[roster] error team_id={team_id}: {e}")
            return None


def daterange_yyyymmdd(days_back: int) -> List[str]:
    today_utc = datetime.now(timezone.utc).date()
    return [(today_utc - timedelta(days=i)).strftime("%Y%m%d") for i in range(days_back)]


def dateforward_yyyymmdd(days_ahead: int) -> List[str]:
    today_utc = datetime.now(timezone.utc).date()
    return [(today_utc + timedelta(days=i)).strftime("%Y%m%d") for i in range(days_ahead + 1)]


def collect_events_by_dates(client: ESPNClient, days_back: int) -> List[Dict[str, Any]]:
    dates = daterange_yyyymmdd(days_back)
    events: List[Dict[str, Any]] = []

    for ds in dates:
        data = client.get_scoreboard(ds)
        if data and "events" in data:
            events.extend(data["events"])
        time.sleep(client.sleep)

    seen = set()
    uniq = []
    for e in events:
        eid = str(e.get("id"))
        if not eid or eid in seen:
            continue
        seen.add(eid)
        uniq.append(e)
    return uniq


def collect_upcoming_events(client: ESPNClient, days_ahead: int = 2) -> List[Dict[str, Any]]:
    dates = dateforward_yyyymmdd(days_ahead)
    events: List[Dict[str, Any]] = []

    for ds in dates:
        data = client.get_scoreboard(ds)
        if data and "events" in data:
            events.extend(data["events"])
        time.sleep(client.sleep)

    seen = set()
    uniq = []
    for e in events:
        eid = str(e.get("id"))
        if not eid or eid in seen:
            continue
        seen.add(eid)
        uniq.append(e)
    return uniq


def parse_scoreboard_event_minimal(event_obj: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    try:
        eid = str(event_obj["id"])
        date_str = event_obj.get("date")
        game_dt = pd.to_datetime(date_str).tz_convert(None) if date_str else pd.NaT

        competition = event_obj["competitions"][0]
        status = competition["status"]["type"]
        completed = bool(status.get("completed", False))

        competitors = competition["competitors"]
        if len(competitors) != 2:
            return None

        teams = []
        for c in competitors:
            team = c["team"]
            teams.append(
                {
                    "event_id": eid,
                    "game_date": game_dt,
                    "team_id": str(team.get("id")),
                    "team": team.get("displayName"),
                    "abbr": team.get("abbreviation"),
                    "home_away": c.get("homeAway"),
                    "score": safe_float(c.get("score")),
                    "winner": bool(c.get("winner", False)),
                    "completed": completed,
                }
            )
        return {"event_id": eid, "game_date": game_dt, "completed": completed, "teams": teams}
    except Exception:
        return None


# -----------------------------
# TEAM stats (existing)
# -----------------------------
def extract_team_stats_from_summary(summary_json: Dict[str, Any]) -> Dict[str, Dict[str, float]]:
    out: Dict[str, Dict[str, float]] = {}
    box = summary_json.get("boxscore", {}) if isinstance(summary_json, dict) else {}
    teams = box.get("teams", [])
    if not isinstance(teams, list):
        return out

    for t in teams:
        try:
            team_info = t.get("team", {})
            team_id = str(team_info.get("id"))
            if not team_id:
                continue

            stats_map: Dict[str, float] = {}
            stats_list = t.get("statistics", [])
            if isinstance(stats_list, list):
                for s in stats_list:
                    name = s.get("name") or s.get("abbreviation")
                    if not name:
                        continue
                    val = s.get("value")
                    if val is None:
                        val = s.get("displayValue")
                    stats_map[name] = safe_float(val)

            out[team_id] = stats_map
        except Exception:
            continue

    return out


def build_team_game_table(client: ESPNClient, events: List[Dict[str, Any]], keep_incomplete: bool = False) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    for ev in events:
        minimal = parse_scoreboard_event_minimal(ev)
        if not minimal:
            continue

        eid = minimal["event_id"]
        game_date = minimal["game_date"]
        completed = minimal["completed"]

        if (not keep_incomplete) and (not completed):
            continue

        summary = client.get_summary(eid) if completed else None
        team_stats = extract_team_stats_from_summary(summary) if summary else {}

        teams = minimal["teams"]
        if len(teams) != 2:
            continue

        a, b = teams[0], teams[1]

        for tm in teams:
            points = tm["score"]
            opp = b if tm["team_id"] == a["team_id"] else a
            opp_points = opp["score"]

            if completed and (pd.isna(points) or pd.isna(opp_points)):
                continue

            r = {
                "event_id": str(eid),
                "game_date": game_date,
                "team_id": tm["team_id"],
                "team": tm["team"],
                "abbr": tm["abbr"],
                "home_away": tm["home_away"],
                "completed": completed,
                "points": points,
                "opp_points": opp_points,
            }

            stats = team_stats.get(tm["team_id"], {})
            for k, v in stats.items():
                r[f"stat_{k}"] = v

            rows.append(r)

        time.sleep(client.sleep)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")
    df["event_id"] = df["event_id"].astype(str)
    df["team_id"] = df["team_id"].astype(str)
    df = df.loc[:, ~df.columns.duplicated()].copy()

    for c in df.columns:
        if c.startswith("stat_") or c in ("points", "opp_points"):
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


# -----------------------------
# PLAYER stats (NEW)
# -----------------------------
def extract_player_boxscore_stats(summary_json: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Extract player rows from ESPN summary boxscore.
    We try to be robust to ESPN structure changes.
    """
    out: List[Dict[str, Any]] = []
    if not isinstance(summary_json, dict):
        return out

    box = summary_json.get("boxscore", {})
    players = box.get("players", [])
    if not isinstance(players, list):
        return out

    for team_block in players:
        try:
            team_info = team_block.get("team", {})
            team_id = str(team_info.get("id") or "")
            team_name = str(team_info.get("displayName") or "")
            abbr = str(team_info.get("abbreviation") or "")

            stats_groups = team_block.get("statistics", [])
            if not isinstance(stats_groups, list):
                continue

            for grp in stats_groups:
                athletes = grp.get("athletes", [])
                if not isinstance(athletes, list):
                    continue

                for a in athletes:
                    athlete = a.get("athlete", {}) or {}
                    player_id = str(athlete.get("id") or "")
                    player_name = str(athlete.get("displayName") or a.get("name") or "").strip()
                    if not player_id and not player_name:
                        continue

                    stats = a.get("stats", [])
                    # ESPN provides display formats; we rely on the "labels"
                    labels = grp.get("labels", [])
                    if not isinstance(labels, list) or not isinstance(stats, list):
                        continue
                    if len(labels) != len(stats):
                        # sometimes "keys" exist
                        keys = grp.get("keys", [])
                        if isinstance(keys, list) and len(keys) == len(stats):
                            labels = keys
                        else:
                            # best-effort: skip mismatched rows
                            continue

                    row = {
                        "team_id": team_id,
                        "team": team_name,
                        "abbr": abbr,
                        "player_id": player_id,
                        "player": player_name,
                    }

                    # Map common labels -> numeric stats
                    for lab, val in zip(labels, stats):
                        lab_u = str(lab).upper()
                        v = safe_float(val)
                        # We keep a few common ones; you can add more mappings if needed
                        if lab_u in ("MIN",):
                            row["minutes"] = v
                        elif lab_u in ("PTS",):
                            row["points"] = v
                        elif lab_u in ("REB", "TRB"):
                            row["rebounds"] = v
                        elif lab_u in ("AST",):
                            row["assists"] = v
                        elif lab_u in ("3PM", "3FGM", "3PTM"):
                            row["threes"] = v

                    out.append(row)
        except Exception:
            continue

    return out


def build_player_game_table(client: ESPNClient, events: List[Dict[str, Any]]) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    for ev in events:
        minimal = parse_scoreboard_event_minimal(ev)
        if not minimal or not minimal["completed"]:
            continue

        eid = minimal["event_id"]
        game_date = minimal["game_date"]
        teams = minimal["teams"]
        if len(teams) != 2:
            continue

        # build event-level context: home/away, opponent ids
        team_ctx = {}
        for tm in teams:
            team_ctx[tm["team_id"]] = {
                "home_away": tm["home_away"],
                "team": tm["team"],
                "abbr": tm["abbr"],
            }
        # opponent map
        if teams[0]["team_id"] != teams[1]["team_id"]:
            opp_map = {
                teams[0]["team_id"]: teams[1]["team_id"],
                teams[1]["team_id"]: teams[0]["team_id"],
            }
        else:
            continue

        summary = client.get_summary(eid)
        if not summary:
            time.sleep(client.sleep)
            continue

        player_rows = extract_player_boxscore_stats(summary)

        for pr in player_rows:
            team_id = str(pr.get("team_id") or "")
            if team_id not in team_ctx:
                continue

            r = {
                "event_id": str(eid),
                "game_date": game_date,
                "team_id": team_id,
                "opp_team_id": opp_map.get(team_id, ""),
                "home_away": team_ctx[team_id]["home_away"],
                "team": team_ctx[team_id]["team"],
                "abbr": team_ctx[team_id]["abbr"],
                "player_id": str(pr.get("player_id") or ""),
                "player": str(pr.get("player") or ""),
            }

            # outcomes
            for k in PROP_STATS.keys():
                if k in pr:
                    r[k] = safe_float(pr.get(k))

            # keep only if we have at least one prop stat
            if not any([pd.notna(r.get(t)) for t in PROP_TARGETS]):
                continue

            rows.append(r)

        time.sleep(client.sleep)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")
    for c in ["points", "rebounds", "assists", "threes", "minutes"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # drop empty IDs but keep names; still usable for joins
    df["player_id"] = df["player_id"].astype(str)
    df["event_id"] = df["event_id"].astype(str)
    df["team_id"] = df["team_id"].astype(str)
    df["opp_team_id"] = df["opp_team_id"].astype(str)

    return df


def make_player_features(player_game: pd.DataFrame, window: int = ROLL_WINDOW) -> Tuple[pd.DataFrame, List[str]]:
    """
    Builds pre-game rolling features per player:
      - last-window averages for target stats + minutes
    Also adds a couple opponent/team context features if available.
    """
    pg = player_game.copy()
    pg = pg.dropna(subset=["player", "team_id", "game_date"]).copy()
    pg = pg.sort_values(["player", "game_date", "event_id"]).reset_index(drop=True)

    # base numeric cols for rolling
    base_roll_cols = []
    for c in ["minutes"] + PROP_TARGETS:
        if c in pg.columns and pd.api.types.is_numeric_dtype(pg[c]):
            base_roll_cols.append(c)

    for c in base_roll_cols:
        pg[f"roll_{c}_l{window}"] = (
            pg.groupby(["player", "team_id"])[c].shift(1).rolling(window).mean()
        )

    # Simple "recent games played" feature
    pg["games_played_l30"] = (
        pg.groupby(["player", "team_id"])["game_date"]
        .apply(lambda s: s.shift(1).rolling(window=30, min_periods=1).count())
        .reset_index(level=[0,1], drop=True)
    )

    feature_cols = [c for c in pg.columns if c.startswith("roll_")] + ["games_played_l30"]

    return pg, feature_cols


# -----------------------------
# OLS prop model (NEW)
# -----------------------------
@dataclass
class OLSModel:
    feature_cols: List[str]
    coef_: np.ndarray  # includes intercept at index 0
    resid_std_: float  # std of residuals

    def predict_mean(self, X: pd.DataFrame) -> np.ndarray:
        Xv = X[self.feature_cols].copy()
        Xv = Xv.replace([np.inf, -np.inf], np.nan)
        Xv = Xv.fillna(Xv.median(numeric_only=True))
        A = np.column_stack([np.ones(len(Xv)), Xv.values.astype(float)])
        return A @ self.coef_


def train_ols_prop_model(df: pd.DataFrame, target: str, feature_cols: List[str]) -> Optional[OLSModel]:
    """
    Train simple OLS projection: target ~ features (pre-game rolling features).
    Returns an OLSModel with residual std for probability pricing.
    """
    d = df.copy()
    d = d.dropna(subset=[target]).copy()

    # require some usable features
    usable = [c for c in feature_cols if c in d.columns and d[c].notna().any()]
    if len(usable) < 3:
        print(f"[props] Not enough features for {target}.")
        return None

    # Keep players with a minimum number of games
    d["player_games"] = d.groupby(["player", "team_id"])[target].transform("count")
    d = d[d["player_games"] >= MIN_PLAYER_GAMES].copy()
    if len(d) < 500:
        print(f"[props] Too little training data for {target}: {len(d)} rows.")
        return None

    # Train/test split by time (like your win model)
    d = d.sort_values("game_date").reset_index(drop=True)
    n = len(d)
    cut = int(n * 0.85)
    train = d.iloc[:cut]
    test = d.iloc[cut:]

    X_train = train[usable].replace([np.inf, -np.inf], np.nan)
    X_train = X_train.fillna(X_train.median(numeric_only=True))
    y_train = train[target].astype(float).values

    A = np.column_stack([np.ones(len(X_train)), X_train.values.astype(float)])
    coef, _, _, _ = np.linalg.lstsq(A, y_train, rcond=None)

    # residual std on test
    X_test = test[usable].replace([np.inf, -np.inf], np.nan)
    X_test = X_test.fillna(X_test.median(numeric_only=True))
    A_test = np.column_stack([np.ones(len(X_test)), X_test.values.astype(float)])
    y_test = test[target].astype(float).values
    y_pred = A_test @ coef
    resid = y_test - y_pred
    resid_std = float(np.nanstd(resid))

    resid_std = max(resid_std, PROP_STD_FLOOR)

    print(f"[props] {target}: trained on {len(train)} / tested on {len(test)}. resid_std={resid_std:.3f}")
    return OLSModel(feature_cols=usable, coef_=coef, resid_std_=resid_std)


# -----------------------------
# Matchup dataset (existing)
# -----------------------------
def make_matchup_dataset(team_game: pd.DataFrame, window: int = ROLL_WINDOW) -> Tuple[pd.DataFrame, List[str]]:
    tg = team_game.copy()
    tg = tg.dropna(subset=["event_id", "team_id", "game_date"]).copy()
    tg = tg.sort_values(["team_id", "game_date", "event_id"]).reset_index(drop=True)

    numeric_cols = [c for c in tg.columns if pd.api.types.is_numeric_dtype(tg[c])]
    exclude = {"points", "opp_points"}
    numeric_cols = [c for c in numeric_cols if c not in exclude]

    for c in numeric_cols + ["points", "opp_points"]:
        tg[f"roll_{c}_l{window}"] = (
            tg.groupby("team_id")[c].shift(1).rolling(window).mean()
        )

    base_cols = ["event_id", "game_date", "team_id", "team", "abbr", "home_away", "points"]
    g = tg[base_cols].copy()
    g = g.sort_values(["event_id", "home_away"])

    pairs = []
    for eid, grp in g.groupby("event_id"):
        if len(grp) != 2:
            continue

        home = grp[grp["home_away"] == "home"]
        away = grp[grp["home_away"] == "away"]
        if len(home) != 1 or len(away) != 1:
            continue

        home = home.iloc[0]
        away = away.iloc[0]

        pairs.append(
            {
                "event_id": str(eid),
                "game_date": home["game_date"],
                "home_team": home["team"],
                "away_team": away["team"],
                "home_id": home["team_id"],
                "away_id": away["team_id"],
                "pts_home": home["points"],
                "pts_away": away["points"],
                "y_home_win": int(home["points"] > away["points"])
                if (not pd.isna(home["points"]) and not pd.isna(away["points"]))
                else np.nan,
            }
        )

    games = pd.DataFrame(pairs)
    if games.empty:
        return games, []

    roll_cols = [c for c in tg.columns if c.startswith("roll_")]
    feats = tg[["event_id", "team_id"] + roll_cols].copy()

    df = games.merge(feats, left_on=["event_id", "home_id"], right_on=["event_id", "team_id"], how="left")
    df = df.drop(columns=["team_id"]).rename(columns={c: f"h_{c}" for c in roll_cols})

    df = df.merge(feats, left_on=["event_id", "away_id"], right_on=["event_id", "team_id"], how="left")
    df = df.drop(columns=["team_id"]).rename(columns={c: f"a_{c}" for c in roll_cols})

    feature_cols: List[str] = []
    for c in roll_cols:
        hc = f"h_{c}"
        ac = f"a_{c}"
        dc = f"d_{c}"
        if hc in df.columns and ac in df.columns:
            df[dc] = df[hc] - df[ac]
            feature_cols.append(dc)

    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")
    df = df.dropna(subset=["game_date"]).sort_values("game_date").reset_index(drop=True)

    return df, feature_cols


# -----------------------------
# Training + calibration (existing)
# -----------------------------
def train_calibrated_model(df: pd.DataFrame, features: List[str]):
    df = df.copy()
    df = df.dropna(subset=["y_home_win"]).copy()
    df["y_home_win"] = df["y_home_win"].astype(int)

    for c in features:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    feats = [c for c in features if df[c].notna().any()]
    if DROP_CONSTANT_FEATURES:
        feats = [c for c in feats if df[c].nunique(dropna=True) > 1]

    if MIN_NON_NULL_RATE > 0:
        coverage = df[feats].notna().mean()
        feats = coverage[coverage >= MIN_NON_NULL_RATE].index.tolist()

    print(f"Features available: {len(features)}")
    print(f"Features used after cleanup/filter: {len(feats)}")

    df = df.sort_values("game_date").reset_index(drop=True)
    n = len(df)
    cut_train = int(n * 0.70)
    cut_cal = int(n * 0.85)

    train_df = df.iloc[:cut_train]
    cal_df = df.iloc[cut_train:cut_cal]
    test_df = df.iloc[cut_cal:]

    imp = SimpleImputer(strategy="median")
    X_train = imp.fit_transform(train_df[feats])
    y_train = train_df["y_home_win"].values

    X_cal = imp.transform(cal_df[feats])
    y_cal = cal_df["y_home_win"].values

    X_test = imp.transform(test_df[feats])
    y_test = test_df["y_home_win"].values

    base = LogisticRegression(
        solver="liblinear",
        max_iter=6000,
        C=0.7,
    )
    base.fit(X_train, y_train)

    X_traincal = np.vstack([X_train, X_cal])
    y_traincal = np.concatenate([y_train, y_cal])

    cal = CalibratedClassifierCV(base, method="sigmoid", cv=3)
    cal.fit(X_traincal, y_traincal)

    p_test = cal.predict_proba(X_test)[:, 1]
    y_hat = (p_test >= 0.5).astype(int)

    print("Test accuracy:", round(accuracy_score(y_test, y_hat), 4))
    print("Test logloss: ", round(log_loss(y_test, p_test), 4))

    return cal, imp, test_df, y_test, p_test, np.array(feats)


# -----------------------------
# Upcoming game helpers
# -----------------------------
def upcoming_matchups_from_events(events: List[Dict[str, Any]]) -> pd.DataFrame:
    rows = []
    for ev in events:
        minimal = parse_scoreboard_event_minimal(ev)
        if not minimal:
            continue
        if minimal["completed"]:
            continue

        teams = minimal["teams"]
        if len(teams) != 2:
            continue
        home = [t for t in teams if t.get("home_away") == "home"]
        away = [t for t in teams if t.get("home_away") == "away"]
        if len(home) != 1 or len(away) != 1:
            continue

        home = home[0]
        away = away[0]
        rows.append({
            "event_id": str(minimal["event_id"]),
            "game_date": minimal["game_date"],
            "home_team": home["team"],
            "away_team": away["team"],
            "home_id": str(home["team_id"]),
            "away_id": str(away["team_id"]),
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")
    return df.sort_values("game_date").reset_index(drop=True)


def extract_roster_players(roster_json: Dict[str, Any]) -> List[Dict[str, str]]:
    """
    Extract (player_id, player_name) from ESPN roster endpoint.
    """
    out = []
    if not isinstance(roster_json, dict):
        return out
    athletes = roster_json.get("athletes", [])
    if not isinstance(athletes, list):
        # sometimes nested: { athletes: [ { items: [...] } ] }
        # try to flatten
        ath_groups = roster_json.get("athletes", [])
        if isinstance(ath_groups, list):
            for g in ath_groups:
                items = g.get("items", [])
                if isinstance(items, list):
                    athletes.extend(items)

    for a in athletes:
        try:
            athlete = a.get("athlete", a)
            pid = str(athlete.get("id") or "")
            name = str(athlete.get("displayName") or athlete.get("fullName") or athlete.get("name") or "").strip()
            if name:
                out.append({"player_id": pid, "player": name})
        except Exception:
            continue
    return out


# -----------------------------
# Main
# -----------------------------
def main():
    client = ESPNClient(sleep=SLEEP_BETWEEN_CALLS)

    print("\n=== 1) Collect historical events ===")
    events_hist = collect_events_by_dates(client, days_back=DEFAULT_N_DAYS_HISTORY)
    print(f"Events collected (raw): {len(events_hist)}")

    print("\n=== 2) Build team-game table (completed games) ===")
    team_game = build_team_game_table(client, events_hist, keep_incomplete=False)
    if team_game.empty:
        print("No completed games found. Exiting.")
        return

    print("team_game shape:", team_game.shape)
    print("team_game date range:", team_game["game_date"].min(), "→", team_game["game_date"].max())

    print("\n=== 3) Build matchup dataset + rolling features (WIN MODEL) ===")
    df, FEATURE_COLS = make_matchup_dataset(team_game, window=ROLL_WINDOW)
    if df.empty or not FEATURE_COLS:
        print("Could not build matchup dataset / no features. Exiting.")
        return

    print("\n=== 4) Train + calibrate WIN model ===")
    win_model, win_imputer, test_df, y_test, p_test, FEATURES_USED = train_calibrated_model(df, FEATURE_COLS)

    out = test_df[["game_date", "home_team", "away_team", "pts_home", "pts_away", "y_home_win"]].copy()
    out["p_home_win"] = np.round(p_test, 3)
    out["fair_american_home"] = out["p_home_win"].apply(to_american_odds).round(1)
    out["fair_american_away"] = (1 - out["p_home_win"]).apply(to_american_odds).round(1)

    latest_day = out["game_date"].max()
    week_start = latest_day - pd.Timedelta(days=7)

    print("\n=== LATEST WEEK (COMPLETED, from TEST SPLIT) ===")
    print(
        out[out["game_date"] >= week_start]
        .sort_values("game_date", ascending=False)
        .head(30)
        .to_string(index=False)
    )

    # -----------------------------
    # PROPS BRANCH (NEW)
    # -----------------------------
    print("\n=== 5) Build player-game table (completed games) ===")
    player_game = build_player_game_table(client, events_hist)
    if player_game.empty:
        print("No player-game rows found. Exiting props branch.")
    else:
        print("player_game shape:", player_game.shape)
        print("player_game date range:", player_game["game_date"].min(), "→", player_game["game_date"].max())

        print("\n=== 6) Build player rolling features ===")
        pg_feat, player_feat_cols = make_player_features(player_game, window=ROLL_WINDOW)
        print("Player feature cols:", len(player_feat_cols))
        print("Example:", player_feat_cols[:10])

        print("\n=== 7) Train prop projection models (your own lines + odds) ===")
        prop_models: Dict[str, OLSModel] = {}
        for tgt in PROP_TARGETS:
            if tgt not in pg_feat.columns:
                continue
            m = train_ols_prop_model(pg_feat, target=tgt, feature_cols=player_feat_cols)
            if m is not None:
                prop_models[tgt] = m

        if not prop_models:
            print("No prop models trained. Exiting props branch.")
        else:
            # -----------------------------
            # UPCOMING predictions (WIN + PROPS)
            # -----------------------------
            print("\n=== 8) Upcoming games (next 2 days) ===")
            upcoming_events = collect_upcoming_events(client, days_ahead=2)
            upcoming_matchups = upcoming_matchups_from_events(upcoming_events)

            if upcoming_matchups.empty:
                print("No upcoming games found in next 2 days.")
            else:
                # WIN preds for upcoming: build matchup rows using latest rolling features
                # We rebuild matchup dataset but keep incomplete games by using team_game history only:
                # For each upcoming event, we need home/away rolling features as-of that date.
                # We'll approximate by using the latest available roll features in df (historical),
                # matched by team id.
                print("\n--- WIN predictions (approx using latest team rolling features) ---")

                # Build latest team roll snapshot from team_game history
                # Recompute team rolling means like training did (same window)
                tg = team_game.copy().sort_values(["team_id", "game_date", "event_id"]).reset_index(drop=True)
                num_cols = [c for c in tg.columns if pd.api.types.is_numeric_dtype(tg[c])]
                exclude = {"points", "opp_points"}
                num_cols = [c for c in num_cols if c not in exclude]

                for c in num_cols + ["points", "opp_points"]:
                    tg[f"roll_{c}_l{ROLL_WINDOW}"] = (
                        tg.groupby("team_id")[c].shift(0).rolling(ROLL_WINDOW).mean()
                    )

                roll_cols = [c for c in tg.columns if c.startswith("roll_")]
                latest_team = (
                    tg.dropna(subset=["team_id", "game_date"])
                      .sort_values("game_date")
                      .groupby("team_id")
                      .tail(1)
                      [["team_id"] + roll_cols]
                      .reset_index(drop=True)
                )

                # Create feature diff rows for upcoming
                up = upcoming_matchups.merge(latest_team, left_on="home_id", right_on="team_id", how="left")
                up = up.drop(columns=["team_id"]).rename(columns={c: f"h_{c}" for c in roll_cols})
                up = up.merge(latest_team, left_on="away_id", right_on="team_id", how="left")
                up = up.drop(columns=["team_id"]).rename(columns={c: f"a_{c}" for c in roll_cols})

                # Build diff features in the same names used in training (d_roll_...)
                X_cols = []
                for c in roll_cols:
                    hc = f"h_{c}"
                    ac = f"a_{c}"
                    dc = f"d_{c}"
                    if hc in up.columns and ac in up.columns:
                        up[dc] = up[hc] - up[ac]
                        X_cols.append(dc)

                # Align to FEATURES_USED and imputer
                # Fill missing columns if needed
                for col in FEATURES_USED:
                    if col not in up.columns:
                        up[col] = np.nan

                X_up = win_imputer.transform(up[list(FEATURES_USED)])
                p_home = win_model.predict_proba(X_up)[:, 1]

                win_out = up[["game_date", "home_team", "away_team"]].copy()
                win_out["p_home_win"] = np.round(p_home, 3)
                win_out["fair_home_ml"] = win_out["p_home_win"].apply(to_american_odds).round(1)
                win_out["fair_away_ml"] = (1 - win_out["p_home_win"]).apply(to_american_odds).round(1)

                print(win_out.sort_values("game_date").to_string(index=False))

                # -----------------------------
                # PROPS for upcoming
                # -----------------------------
                print("\n--- PROP predictions (your own lines + fair odds) ---")

                # Latest player feature snapshot (use last available row per player/team)
                pg_latest = (
                    pg_feat.dropna(subset=["player", "team_id", "game_date"])
                          .sort_values("game_date")
                          .groupby(["player", "team_id"])
                          .tail(1)
                          .reset_index(drop=True)
                )

                prop_rows = []
                for _, g in upcoming_matchups.iterrows():
                    home_id = str(g["home_id"])
                    away_id = str(g["away_id"])
                    game_date = g["game_date"]
                    home_team = g["home_team"]
                    away_team = g["away_team"]

                    # Pull rosters
                    home_roster = client.get_team_roster(home_id) or {}
                    away_roster = client.get_team_roster(away_id) or {}
                    time.sleep(client.sleep)

                    home_players = extract_roster_players(home_roster)
                    away_players = extract_roster_players(away_roster)

                    # Build candidate pool: players on roster with history in pg_latest
                    def candidates(team_id: str, roster_players: List[Dict[str, str]]) -> pd.DataFrame:
                        names = [p["player"] for p in roster_players if p.get("player")]
                        # Match by name (most robust without consistent IDs)
                        cand = pg_latest[(pg_latest["team_id"] == team_id) & (pg_latest["player"].isin(names))].copy()
                        if cand.empty:
                            # fallback: try contains match (best-effort)
                            cand = pg_latest[(pg_latest["team_id"] == team_id)].copy()
                            cand["__roster_hit"] = cand["player"].apply(
                                lambda x: any(str(x).lower() == str(n).lower() for n in names)
                            )
                            cand = cand[cand["__roster_hit"]].copy()
                            cand = cand.drop(columns=["__roster_hit"], errors="ignore")
                        return cand

                    home_cand = candidates(home_id, home_players)
                    away_cand = candidates(away_id, away_players)

                    # Rank by recent minutes
                    min_col = f"roll_minutes_l{ROLL_WINDOW}"
                    if min_col in home_cand.columns:
                        home_cand = home_cand.sort_values(min_col, ascending=False)
                    if min_col in away_cand.columns:
                        away_cand = away_cand.sort_values(min_col, ascending=False)

                    home_cand = home_cand.head(TOP_PLAYERS_PER_TEAM)
                    away_cand = away_cand.head(TOP_PLAYERS_PER_TEAM)

                    for side, team_name, cand in [("HOME", home_team, home_cand), ("AWAY", away_team, away_cand)]:
                        if cand.empty:
                            continue

                        for _, pr in cand.iterrows():
                            player_name = pr["player"]

                            for tgt, model in prop_models.items():
                                # build a single-row X
                                X_row = pr.to_frame().T
                                mean = float(model.predict_mean(X_row)[0])
                                std = float(model.resid_std_)

                                # Your own “fair line” = projected mean rounded to 0.5
                                fair_line = round_to_step(mean, PROP_LINE_ROUND_TO)

                                p_over = prob_over_normal(mean, std, fair_line)
                                p_under = 1.0 - p_over

                                prop_rows.append({
                                    "game_date": game_date,
                                    "matchup": f"{away_team} @ {home_team}",
                                    "side": side,
                                    "team": team_name,
                                    "player": player_name,
                                    "prop": tgt,
                                    "proj_mean": round(mean, 2),
                                    "fair_line": fair_line,
                                    "p_over": round(p_over, 3),
                                    "p_under": round(p_under, 3),
                                    "fair_over_odds": round(to_american_odds(p_over), 1),
                                    "fair_under_odds": round(to_american_odds(p_under), 1),
                                })

                props_out = pd.DataFrame(prop_rows)
                if props_out.empty:
                    print("No props generated (roster match may have failed or no history).")
                else:
                    props_out = props_out.sort_values(["game_date", "matchup", "prop", "side", "team", "player"])
                    # Print a manageable chunk
                    print(props_out.head(120).to_string(index=False))

    print("\n=== Done ✅ ===")


if __name__ == "__main__":
    main()



=== 1) Collect historical events ===
Events collected (raw): 782

=== 2) Build team-game table (completed games) ===
team_game shape: (1524, 34)
team_game date range: 2025-10-02 16:00:00 → 2026-01-27 02:30:00

=== 3) Build matchup dataset + rolling features (WIN MODEL) ===

=== 4) Train + calibrate WIN model ===
Features available: 28
Features used after cleanup/filter: 24
Test accuracy: 0.5391
Test logloss:  0.6875

=== LATEST WEEK (COMPLETED, from TEST SPLIT) ===
          game_date              home_team              away_team  pts_home  pts_away  y_home_win  p_home_win  fair_american_home  fair_american_away
2026-01-27 02:30:00 Minnesota Timberwolves  Golden State Warriors     108.0      83.0           1       0.408               145.1              -145.1
2026-01-27 01:00:00         Boston Celtics Portland Trail Blazers     102.0      94.0           1       0.586              -141.5               141.5
2026-01-27 01:00:00        Houston Rockets      Memphis Grizzlies     108.0    

/var/folders/pj/1wf8h2rx47nf6pwhy55jvs2c0000gn/T/ipykernel_53005/4247751951.py:566: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Xv = Xv.replace([np.inf, -np.inf], np.nan)
/var/folders/pj/1wf8h2rx47nf6pwhy55jvs2c0000gn/T/ipykernel_53005/4247751951.py:566: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Xv = Xv.replace([np.inf, -np.inf], np.nan)
/var/folders/pj/1wf8h2rx47nf6pwhy55jvs2c0000gn/T/ipykernel_53005/4247751951.py:566: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To

          game_date                                  matchup side                team                   player     prop  proj_mean  fair_line  p_over  p_under  fair_over_odds  fair_under_odds
2026-01-29 00:00:00           Chicago Bulls @ Indiana Pacers AWAY       Chicago Bulls              Ayo Dosunmu  assists       2.77        3.0   0.455    0.545           119.9           -119.9
2026-01-29 00:00:00           Chicago Bulls @ Indiana Pacers AWAY       Chicago Bulls               Coby White  assists       4.83        5.0   0.466    0.534           114.5           -114.5
2026-01-29 00:00:00           Chicago Bulls @ Indiana Pacers AWAY       Chicago Bulls              Isaac Okoro  assists       1.86        2.0   0.473    0.527           111.6           -111.6
2026-01-29 00:00:00           Chicago Bulls @ Indiana Pacers AWAY       Chicago Bulls              Jalen Smith  assists       1.77        2.0   0.455    0.545           119.9           -119.9
2026-01-29 00:00:00           Chicago Bu

/var/folders/pj/1wf8h2rx47nf6pwhy55jvs2c0000gn/T/ipykernel_53005/4247751951.py:566: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Xv = Xv.replace([np.inf, -np.inf], np.nan)
/var/folders/pj/1wf8h2rx47nf6pwhy55jvs2c0000gn/T/ipykernel_53005/4247751951.py:566: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Xv = Xv.replace([np.inf, -np.inf], np.nan)
/var/folders/pj/1wf8h2rx47nf6pwhy55jvs2c0000gn/T/ipykernel_53005/4247751951.py:566: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To